# Pascal VOC 2007+2012 - ConvNeXt Faster R-CNN Selective QAT

Notebook này chạy trên Kaggle để tạo benchmark cho pipeline ConvNeXt selective QAT trên dataset Pascal VOC 2007+2012:

1. Clone repo EchteAI.
2. Tìm dataset Kaggle `vijayabhaskar96/pascal-voc-2007-and-2012`.
3. Convert Pascal VOC XML sang COCO JSON.
4. Train FP32 từng epoch, mỗi epoch tự validation + benchmark 100 ảnh + lưu checkpoint/result.
5. Vẽ biểu đồ hội tụ FP32 theo mAP, loss và latency.
6. Khi FP32 hội tụ hoặc chạm số epoch tối đa, bắt đầu train QAT.
7. Train QAT epoch 1, convert INT8 tạm và benchmark FP32 vs INT8.
8. Resume QAT epoch 2, convert INT8 final và benchmark lại.

Lưu ý: checkpoint SeaDronesSee 6 lớp không dùng trực tiếp được cho Pascal VOC 21 lớp. Notebook này train lại FP32 trên Pascal VOC trước rồi mới bù lỗi QAT.

In [ ]:
# Cell 1 - Cấu hình người dùng
from pathlib import Path

REPO_URL = 'https://github.com/NguyenDucThang-tb/EchteAI.git'
REPO_BRANCH = 'main'
DATASET_HANDLE = 'vijayabhaskar96/pascal-voc-2007-and-2012'

ROOT = Path('/kaggle/working')
WORK = ROOT / 'pascal_voc_convnext_qat'
OUTPUT = WORK / 'checkpoints'
LOGS = WORK / 'logs'
COCO_ROOT = WORK / 'coco'

VARIANT = 'M3'

# FP32 sẽ train từng epoch cho tới khi hội tụ hoặc chạm max epoch.
FP32_MAX_EPOCHS = 10
FP32_PATIENCE = 2
FP32_MIN_DELTA = 0.002
FORCE_START_QAT = False  # True nếu muốn bắt đầu QAT dù FP32 chưa hội tụ.

# QAT chạy đúng 2 epoch để so sánh epoch 1 và epoch 2.
QAT_TOTAL_EPOCHS = 2
BENCHMARK_IMAGES = 100

# Để None nếu muốn chạy toàn bộ VOC. Đặt 500/1000 để smoke test nhanh.
TRAIN_LIMIT = None

# Batch size là per-GPU khi chạy DDP. Kaggle T4 x2 => global batch = batch_size * 2.
FP32_BATCH_SIZE = 2
QAT_BATCH_SIZE = 1

# Pascal VOC ít vật thể nhỏ hơn SeaDronesSee, dùng size vừa phải để Kaggle chạy nhanh hơn.
MODEL_MIN_SIZE = 640
MODEL_TRAIN_MIN_SIZES = [512, 608, 640]
MODEL_MAX_SIZE = 1024

# Upload kết quả thành Kaggle Dataset ở cuối. Điền username nếu muốn upload.
KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'
CHECKPOINT_DATASET = f'{KAGGLE_USERNAME}/echteai-pascal-voc-convnext-qat'

OUTPUT.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)
COCO_ROOT.mkdir(parents=True, exist_ok=True)
print('WORK:', WORK)
print('OUTPUT:', OUTPUT)
print('FP32 convergence:', {'max_epochs': FP32_MAX_EPOCHS, 'patience': FP32_PATIENCE, 'min_delta': FP32_MIN_DELTA})

In [ ]:
# Cell 2 - Clone repo và cài dependencies
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('/kaggle/working')
os.chdir(ROOT)
REPO = ROOT / 'EchteAI'
if not REPO.exists():
    print(f'Cloning {REPO_URL} -> {REPO}', flush=True)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], check=True, cwd=ROOT)
else:
    print('Repo already exists:', REPO, flush=True)
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], check=False, cwd=REPO)
    subprocess.run(['git', 'checkout', REPO_BRANCH], check=True, cwd=REPO)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], check=False, cwd=REPO)

os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]'], check=True, cwd=REPO)

import torch
print('Repo:', REPO)
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

In [ ]:
# Cell 3 - Helper chạy lệnh và đọc checkpoint
import datetime
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

os.chdir(REPO)

def run_and_log(command, log_path, cwd=REPO):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', ' '.join(map(str, command)), flush=True)
    print('Persistent log:', log_path, flush=True)
    print('Started:', datetime.datetime.now().isoformat(timespec='seconds'), flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'\n===== START {datetime.datetime.now().isoformat()} =====\n')
        log_file.write(' '.join(map(str, command)) + '\n')
        log_file.flush()
        process = subprocess.Popen(
            [str(x) for x in command],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=str(cwd),
            env=env,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
        code = process.wait()
        log_file.write(f'===== END code={code} {datetime.datetime.now().isoformat()} =====\n')
    if code != 0:
        raise subprocess.CalledProcessError(code, command)


def checkpoint_epoch(path):
    path = Path(path)
    if not path.exists():
        return 0
    payload = torch.load(path, map_location='cpu', weights_only=False)
    return int(payload.get('epoch', 0)) if isinstance(payload, dict) else 0


def checkpoint_size_mb(path):
    path = Path(path)
    return path.stat().st_size / 2**20 if path.exists() else 0.0


def print_checkpoint_summary(path):
    path = Path(path)
    for item in sorted(path.glob('*')):
        if item.is_file():
            print(f'{item.name:35s} {item.stat().st_size / 2**20:8.2f} MB')

In [ ]:
# Cell 4 - Tìm hoặc tải dataset Pascal VOC trên Kaggle
import os
from pathlib import Path

import kagglehub


def find_dataset_root():
    candidates = [
        Path('/kaggle/input/pascal-voc-2007-and-2012'),
        Path('/kaggle/input') / DATASET_HANDLE.split('/')[-1],
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    print('Dataset not attached in /kaggle/input; downloading with kagglehub...', flush=True)
    return Path(kagglehub.dataset_download(DATASET_HANDLE))

DATA_DOWNLOAD = find_dataset_root()
print('DATA_DOWNLOAD:', DATA_DOWNLOAD)
print('Top-level files/folders:')
for item in sorted(DATA_DOWNLOAD.iterdir())[:30]:
    print(' ', item)

In [ ]:
# Cell 5 - Convert Pascal VOC XML sang COCO JSON
import json
import random
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
CLASS_TO_ID = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}


def find_voc_roots(root):
    roots = []
    for ann_dir in root.rglob('Annotations'):
        voc_root = ann_dir.parent
        if (voc_root / 'JPEGImages').exists():
            roots.append(voc_root)
    roots = sorted(set(roots))
    assert roots, f'Không tìm thấy thư mục Pascal VOC trong {root}'
    return roots


def read_split(voc_root, split):
    split_file = voc_root / 'ImageSets' / 'Main' / f'{split}.txt'
    if not split_file.exists():
        return []
    return [line.strip().split()[0] for line in split_file.read_text().splitlines() if line.strip()]


def collect_records(voc_roots):
    train_records, val_records, all_records = [], [], []
    for voc_root in voc_roots:
        year = voc_root.name
        train_ids = read_split(voc_root, 'trainval') or read_split(voc_root, 'train')
        val_ids = read_split(voc_root, 'test') or read_split(voc_root, 'val')
        ann_dir = voc_root / 'Annotations'
        image_dir = voc_root / 'JPEGImages'
        if not train_ids:
            train_ids = [p.stem for p in ann_dir.glob('*.xml')]
        for image_id in train_ids:
            xml = ann_dir / f'{image_id}.xml'
            jpg = image_dir / f'{image_id}.jpg'
            if xml.exists() and jpg.exists():
                train_records.append((year, image_id, xml, jpg))
                all_records.append((year, image_id, xml, jpg))
        for image_id in val_ids:
            xml = ann_dir / f'{image_id}.xml'
            jpg = image_dir / f'{image_id}.jpg'
            if xml.exists() and jpg.exists():
                val_records.append((year, image_id, xml, jpg))
                all_records.append((year, image_id, xml, jpg))
    if not val_records:
        random.seed(42)
        all_unique = sorted(set(all_records), key=lambda x: (x[0], x[1]))
        random.shuffle(all_unique)
        cut = max(1, int(0.1 * len(all_unique)))
        val_records = all_unique[:cut]
        train_records = all_unique[cut:]
    return train_records, val_records


def voc_record_to_coco(records, image_root, output_json):
    images, annotations = [], []
    ann_id = 1
    image_id = 1
    for year, stem, xml_path, image_path in records:
        try:
            with Image.open(image_path) as img:
                width, height = img.size
        except Exception:
            root = ET.parse(xml_path).getroot()
            size = root.find('size')
            width = int(size.findtext('width'))
            height = int(size.findtext('height'))
        rel_file = image_path.relative_to(image_root).as_posix()
        images.append({'id': image_id, 'file_name': rel_file, 'width': width, 'height': height})
        root = ET.parse(xml_path).getroot()
        for obj in root.findall('object'):
            name = obj.findtext('name')
            difficult = int(obj.findtext('difficult') or 0)
            if name not in CLASS_TO_ID:
                continue
            box = obj.find('bndbox')
            xmin = max(0.0, float(box.findtext('xmin')) - 1.0)
            ymin = max(0.0, float(box.findtext('ymin')) - 1.0)
            xmax = min(float(width), float(box.findtext('xmax')))
            ymax = min(float(height), float(box.findtext('ymax')))
            w = max(0.0, xmax - xmin)
            h = max(0.0, ymax - ymin)
            if w <= 0 or h <= 0:
                continue
            annotations.append({
                'id': ann_id,
                'image_id': image_id,
                'category_id': CLASS_TO_ID[name],
                'bbox': [xmin, ymin, w, h],
                'area': w * h,
                'iscrowd': 0,
                'difficult': difficult,
            })
            ann_id += 1
        image_id += 1
    categories = [{'id': i + 1, 'name': name, 'supercategory': 'object'} for i, name in enumerate(VOC_CLASSES)]
    payload = {'images': images, 'annotations': annotations, 'categories': categories}
    output_json = Path(output_json)
    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(json.dumps(payload), encoding='utf-8')
    return payload

voc_roots = find_voc_roots(DATA_DOWNLOAD)
print('VOC roots:')
for root in voc_roots:
    print(' ', root)
train_records, val_records = collect_records(voc_roots)
IMAGE_ROOT = DATA_DOWNLOAD
TRAIN_JSON = COCO_ROOT / 'instances_train.json'
VAL_JSON = COCO_ROOT / 'instances_val.json'
train_coco = voc_record_to_coco(train_records, IMAGE_ROOT, TRAIN_JSON)
val_coco = voc_record_to_coco(val_records, IMAGE_ROOT, VAL_JSON)
print('Train images:', len(train_coco['images']), 'annotations:', len(train_coco['annotations']))
print('Val images:', len(val_coco['images']), 'annotations:', len(val_coco['annotations']))
print('IMAGE_ROOT:', IMAGE_ROOT)
print('TRAIN_JSON:', TRAIN_JSON)
print('VAL_JSON:', VAL_JSON)

In [ ]:
# Cell 6 - Tạo runtime config cho Pascal VOC
import yaml
from pathlib import Path

base = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
base['dataset'].update({
    'train_images': str(IMAGE_ROOT),
    'train_annotations': str(TRAIN_JSON),
    'val_images': str(IMAGE_ROOT),
    'val_annotations': str(VAL_JSON),
    'test_images': str(IMAGE_ROOT),
    'test_annotations': str(VAL_JSON),
    'ignore_category_ids': [],
    'num_classes': 21,
    'workers': 2,
})
base['model'].update({
    'backbone': 'convnext_tiny',
    'pretrained_backbone': True,
    'trainable_backbone_layers': 4,
    'min_size': MODEL_MIN_SIZE,
    'train_min_sizes': MODEL_TRAIN_MIN_SIZES,
    'max_size': MODEL_MAX_SIZE,
    'anchor_sizes': 'auto',
    'anchor_statistics_min_size': MODEL_MIN_SIZE,
})
base['training'].update({
    'fp32_batch_size': FP32_BATCH_SIZE,
    'qat_batch_size': QAT_BATCH_SIZE,
    'fp32_epochs': FP32_MAX_EPOCHS,
    'qat_epochs': QAT_TOTAL_EPOCHS,
    'epoch_benchmark_images': BENCHMARK_IMAGES,
    'print_frequency': 50,
    'warmup_iterations': 500,
})
# QAT 2 epoch: epoch 1 full fake-quant, epoch 2 frozen observer để convert final.
base['quantization']['variant'] = VARIANT
base['quantization']['backend'] = 'auto'
base['quantization']['calibration_images'] = 256
base['quantization']['weight_only_warmup_epochs'] = 0
base['quantization']['observer_freeze_epochs'] = 1
base['output'] = {
    'directory': str(OUTPUT),
    'fp32_best': str(OUTPUT / 'fp32_best.pt'),
    'fp32_last': str(OUTPUT / 'fp32_last.pt'),
    'qat_best': str(OUTPUT / 'qat_best.pt'),
    'qat_last': str(OUTPUT / 'qat_last.pt'),
    'int8_model': str(OUTPUT / 'selective_int8.pt'),
    'evaluation_json': str(OUTPUT / 'evaluation.json'),
    'benchmark_json': str(OUTPUT / 'benchmark.json'),
    'epoch_benchmarks': str(OUTPUT / 'epoch_benchmarks.json'),
}
base['benchmark'] = {'warmup_iterations': 10, 'iterations': 50, 'num_threads': 1}
RUNTIME_CONFIG = WORK / 'runtime_pascal_voc.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False), encoding='utf-8')
print('Runtime config:', RUNTIME_CONFIG)
print(RUNTIME_CONFIG.read_text())

In [ ]:
# Cell 7 - Train FP32 từng epoch, benchmark và dừng khi hội tụ
import json
import shutil
from pathlib import Path

import torch

FP32_HISTORY_JSON = OUTPUT / 'fp32_convergence_history.json'
FP32_CONVERGED_FLAG = OUTPUT / 'fp32_converged.json'


def read_checkpoint_metrics(path):
    path = Path(path)
    if not path.exists():
        return {}
    payload = torch.load(path, map_location='cpu', weights_only=False)
    return payload.get('metrics', {}) if isinstance(payload, dict) else {}


def load_epoch_benchmarks():
    path = OUTPUT / 'epoch_benchmarks.json'
    if not path.exists():
        return []
    return json.loads(path.read_text())


def fp32_history_record(epoch):
    metrics = read_checkpoint_metrics(OUTPUT / 'fp32_last.pt')
    train = metrics.get('train', {})
    benchmark = metrics.get('benchmark', {})
    return {
        'epoch': int(epoch),
        'train_loss': train.get('loss'),
        'map_50_95': metrics.get('map_50_95'),
        'map_50': metrics.get('map_50'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'accuracy': metrics.get('accuracy'),
        'mean_iou': metrics.get('mean_iou'),
        'benchmark_latency_ms': benchmark.get('latency_ms_per_image'),
        'benchmark_fps': benchmark.get('fps'),
    }


def update_fp32_history(epoch):
    history = []
    if FP32_HISTORY_JSON.exists():
        history = json.loads(FP32_HISTORY_JSON.read_text())
    history = [item for item in history if int(item.get('epoch', -1)) != int(epoch)]
    history.append(fp32_history_record(epoch))
    history = sorted(history, key=lambda item: item['epoch'])
    FP32_HISTORY_JSON.write_text(json.dumps(history, indent=2), encoding='utf-8')
    return history


def convergence_status(history, patience=2, min_delta=0.002):
    valid = [item for item in history if item.get('map_50_95') is not None]
    if not valid:
        return False, 'no validation metric yet'
    best = -1.0
    stale = 0
    best_epoch = valid[0]['epoch']
    for item in valid:
        value = float(item['map_50_95'])
        if value > best + float(min_delta):
            best = value
            best_epoch = int(item['epoch'])
            stale = 0
        else:
            stale += 1
    converged = stale >= int(patience)
    reason = f'best_epoch={best_epoch} best_map={best:.4f} stale_epochs={stale}/{patience}'
    return converged, reason


def run_fp32_one_epoch():
    fp32_last = OUTPUT / 'fp32_last.pt'
    if torch.cuda.device_count() >= 2:
        command = [
            sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
            'scripts/train_fp32_ddp.py', '--config', str(RUNTIME_CONFIG), '--epochs-this-run', '1',
            '--no-find-unused-parameters',
        ]
        if fp32_last.exists():
            command += ['--resume', str(fp32_last)]
    else:
        command = [sys.executable, '-u', 'scripts/train_fp32.py', '--config', str(RUNTIME_CONFIG), '--epochs-this-run', '1']
        if fp32_last.exists():
            command += ['--resume', str(fp32_last)]
    if TRAIN_LIMIT is not None:
        command += ['--limit', str(TRAIN_LIMIT)]
    run_and_log(command, LOGS / 'fp32_train_until_converged.log', cwd=REPO)

fp32_last = OUTPUT / 'fp32_last.pt'
fp32_epoch = checkpoint_epoch(fp32_last)
print(f'Current FP32 epoch: {fp32_epoch}/{FP32_MAX_EPOCHS}')
print('GPU count:', torch.cuda.device_count(), '| expected DDP when >=2 GPUs')

while fp32_epoch < FP32_MAX_EPOCHS:
    print(f'===== FP32 epoch {fp32_epoch + 1}/{FP32_MAX_EPOCHS} =====', flush=True)
    run_fp32_one_epoch()
    fp32_epoch = checkpoint_epoch(fp32_last)
    snapshot = OUTPUT / f'fp32_epoch_{fp32_epoch:02d}.pt'
    shutil.copy2(fp32_last, snapshot)
    history = update_fp32_history(fp32_epoch)
    converged, reason = convergence_status(history, FP32_PATIENCE, FP32_MIN_DELTA)
    status = {
        'converged': bool(converged),
        'reason': reason,
        'epoch': int(fp32_epoch),
        'max_epochs': int(FP32_MAX_EPOCHS),
        'patience': int(FP32_PATIENCE),
        'min_delta': float(FP32_MIN_DELTA),
    }
    FP32_CONVERGED_FLAG.write_text(json.dumps(status, indent=2), encoding='utf-8')
    print('Saved FP32 snapshot:', snapshot)
    print('Convergence:', status)
    print_checkpoint_summary(OUTPUT)
    if converged:
        print('FP32 đã hội tụ theo tiêu chí đã đặt; dừng FP32 và chuyển sang QAT.')
        break

if fp32_epoch >= FP32_MAX_EPOCHS:
    history = update_fp32_history(fp32_epoch)
    converged, reason = convergence_status(history, FP32_PATIENCE, FP32_MIN_DELTA)
    status = {
        'converged': bool(converged),
        'reason': reason + ' | reached max epoch',
        'epoch': int(fp32_epoch),
        'max_epochs': int(FP32_MAX_EPOCHS),
        'patience': int(FP32_PATIENCE),
        'min_delta': float(FP32_MIN_DELTA),
    }
    FP32_CONVERGED_FLAG.write_text(json.dumps(status, indent=2), encoding='utf-8')
    print('Reached FP32 max epoch; QAT is allowed to start.')

In [ ]:
# Cell 8 - Vẽ biểu đồ hội tụ FP32
import json
from pathlib import Path

import matplotlib.pyplot as plt

history_path = OUTPUT / 'fp32_convergence_history.json'
assert history_path.exists(), 'Chưa có fp32_convergence_history.json; hãy chạy cell train FP32 trước'
history = json.loads(history_path.read_text())
epochs = [item['epoch'] for item in history]
map_50_95 = [item.get('map_50_95') for item in history]
map_50 = [item.get('map_50') for item in history]
losses = [item.get('train_loss') for item in history]
latencies = [item.get('benchmark_latency_ms') for item in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(epochs, map_50_95, marker='o', label='mAP@50:95')
axes[0].plot(epochs, map_50, marker='o', label='mAP@50')
axes[0].set_title('FP32 validation mAP')
axes[0].set_xlabel('Epoch')
axes[0].grid(True)
axes[0].legend()

axes[1].plot(epochs, losses, marker='o', color='tab:red')
axes[1].set_title('FP32 train loss')
axes[1].set_xlabel('Epoch')
axes[1].grid(True)

axes[2].plot(epochs, latencies, marker='o', color='tab:green')
axes[2].set_title(f'FP32 benchmark latency ({BENCHMARK_IMAGES} ảnh)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('ms/image')
axes[2].grid(True)

plt.tight_layout()
PLOT_PATH = OUTPUT / 'fp32_convergence.png'
fig.savefig(PLOT_PATH, dpi=160)
print('Saved plot:', PLOT_PATH)
status_path = OUTPUT / 'fp32_converged.json'
if status_path.exists():
    print(json.dumps(json.loads(status_path.read_text()), indent=2))
plt.show()

In [ ]:
# Cell 9 - Train QAT epoch 1 sau khi FP32 hội tụ
import json
import shutil
import torch

status_path = OUTPUT / 'fp32_converged.json'
assert status_path.exists(), 'Hãy chạy cell train FP32 đến khi có fp32_converged.json trước'
status = json.loads(status_path.read_text())
fp32_ready = bool(status.get('converged')) or int(status.get('epoch', 0)) >= int(status.get('max_epochs', FP32_MAX_EPOCHS))
assert fp32_ready or FORCE_START_QAT, f'FP32 chưa hội tụ: {status}. Đặt FORCE_START_QAT=True nếu vẫn muốn chạy QAT.'

qat_last = OUTPUT / 'qat_last.pt'
qat_epoch = checkpoint_epoch(qat_last)
print(f'Current QAT epoch: {qat_epoch}/{QAT_TOTAL_EPOCHS}')
assert (OUTPUT / 'fp32_best.pt').exists(), 'Cần fp32_best.pt trước khi train QAT'
if qat_epoch < 1:
    if torch.cuda.device_count() >= 2:
        command = [
            sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
            'scripts/train_qat_ddp.py',
            '--config', str(RUNTIME_CONFIG),
            '--fp32-checkpoint', str(OUTPUT / 'fp32_best.pt'),
            '--variant', VARIANT,
            '--epochs-this-run', '1',
            '--no-find-unused-parameters',
        ]
    else:
        command = [
            sys.executable, '-u', 'scripts/train_qat.py',
            '--config', str(RUNTIME_CONFIG),
            '--fp32-checkpoint', str(OUTPUT / 'fp32_best.pt'),
            '--variant', VARIANT,
            '--epochs-this-run', '1',
        ]
    if TRAIN_LIMIT is not None:
        command += ['--limit', str(TRAIN_LIMIT)]
    run_and_log(command, LOGS / 'qat_epoch1.log', cwd=REPO)
    shutil.copy2(OUTPUT / 'qat_last.pt', OUTPUT / 'qat_epoch_01.pt')
else:
    print('QAT epoch 1 already exists; skip.')
print_checkpoint_summary(OUTPUT)

In [ ]:
# Cell 10 - Convert tạm QAT epoch 1 sang INT8 để benchmark 100 ảnh
# Epoch 1 chưa phải final frozen epoch của lịch 2 epoch, nhưng vẫn convert được để lấy benchmark trung gian.
from pipelines.convnext_qat.checkpoint import load_checkpoint, save_checkpoint
from pipelines.convnext_qat.config import load_config, quantized_modules_for_variant
from pipelines.convnext_qat.models import build_fasterrcnn_convnext
from pipelines.convnext_qat.quantization import convert_selective_qat, prepare_selective_qat, set_qat_phase

config = load_config(str(RUNTIME_CONFIG), require_dataset=True)
qat_epoch1 = OUTPUT / 'qat_last.pt'
int8_epoch1 = OUTPUT / 'selective_int8_epoch1.pt'
assert qat_epoch1.exists(), qat_epoch1
payload = torch.load(qat_epoch1, map_location='cpu', weights_only=False)
metadata = payload.get('extra', {}) if isinstance(payload, dict) else {}
variant = str(metadata.get('variant', VARIANT)).upper()
backend = metadata.get('backend', config['quantization'].get('backend', 'auto'))
quantized_modules = metadata.get('quantized_modules', quantized_modules_for_variant(config, variant))
model = build_fasterrcnn_convnext(config)
model = prepare_selective_qat(model, variant, backend, quantized_modules=quantized_modules)
load_checkpoint(qat_epoch1, model, map_location='cpu', strict=True)
set_qat_phase(model, 'frozen')
int8_model = convert_selective_qat(model.to('cpu'))
save_checkpoint(
    int8_epoch1,
    int8_model,
    metrics={'source_qat_epoch': checkpoint_epoch(qat_epoch1)},
    extra={'variant': variant, 'backend': backend, 'format': 'selective_int8', 'quantized_modules': quantized_modules or []},
)
print('Saved INT8 epoch 1:', int8_epoch1, f'{checkpoint_size_mb(int8_epoch1):.2f} MB')

In [ ]:
# Cell 11 - Benchmark FP32 vs INT8 epoch 1 trên 100 ảnh
BENCH1_JSON = OUTPUT / 'fp32_vs_int8_epoch1_100.json'
command = [
    sys.executable, '-u', 'scripts/compare_fp32_int8.py',
    '--config', str(RUNTIME_CONFIG),
    '--fp32-checkpoint', str(OUTPUT / 'fp32_best.pt'),
    '--int8-checkpoint', str(OUTPUT / 'selective_int8_epoch1.pt'),
    '--images', str(BENCHMARK_IMAGES),
    '--threads', '1',
    '--output', str(BENCH1_JSON),
]
run_and_log(command, LOGS / 'benchmark_epoch1.log', cwd=REPO)
print('Saved:', BENCH1_JSON)

In [ ]:
# Cell 12 - Resume QAT epoch 2 và convert INT8 final
qat_last = OUTPUT / 'qat_last.pt'
qat_epoch = checkpoint_epoch(qat_last)
print(f'Current QAT epoch: {qat_epoch}/{QAT_TOTAL_EPOCHS}')
if qat_epoch < 2:
    if torch.cuda.device_count() >= 2:
        command = [
            sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
            'scripts/train_qat_ddp.py',
            '--config', str(RUNTIME_CONFIG),
            '--fp32-checkpoint', str(OUTPUT / 'fp32_best.pt'),
            '--resume', str(qat_last),
            '--variant', VARIANT,
            '--epochs-this-run', '1',
            '--no-find-unused-parameters',
        ]
    else:
        command = [
            sys.executable, '-u', 'scripts/train_qat.py',
            '--config', str(RUNTIME_CONFIG),
            '--fp32-checkpoint', str(OUTPUT / 'fp32_best.pt'),
            '--resume', str(qat_last),
            '--variant', VARIANT,
            '--epochs-this-run', '1',
        ]
    if TRAIN_LIMIT is not None:
        command += ['--limit', str(TRAIN_LIMIT)]
    run_and_log(command, LOGS / 'qat_epoch2.log', cwd=REPO)
    shutil.copy2(OUTPUT / 'qat_last.pt', OUTPUT / 'qat_epoch_02.pt')
else:
    print('QAT epoch 2 already exists; skip.')
print_checkpoint_summary(OUTPUT)
assert (OUTPUT / 'selective_int8.pt').exists(), 'Final selective_int8.pt chưa được tạo; kiểm tra log QAT epoch 2'

In [ ]:
# Cell 13 - Benchmark FP32 vs INT8 epoch 2/final trên 100 ảnh
BENCH2_JSON = OUTPUT / 'fp32_vs_int8_epoch2_100.json'
command = [
    sys.executable, '-u', 'scripts/compare_fp32_int8.py',
    '--config', str(RUNTIME_CONFIG),
    '--fp32-checkpoint', str(OUTPUT / 'fp32_best.pt'),
    '--int8-checkpoint', str(OUTPUT / 'selective_int8.pt'),
    '--images', str(BENCHMARK_IMAGES),
    '--threads', '1',
    '--output', str(BENCH2_JSON),
]
run_and_log(command, LOGS / 'benchmark_epoch2.log', cwd=REPO)
print('Saved:', BENCH2_JSON)

In [ ]:
# Cell 14 - Tổng hợp kết quả benchmark và vẽ biểu đồ QAT
import json
from pathlib import Path

import matplotlib.pyplot as plt

summary = {}
for name, path in [
    ('qat_epoch1', OUTPUT / 'fp32_vs_int8_epoch1_100.json'),
    ('qat_epoch2', OUTPUT / 'fp32_vs_int8_epoch2_100.json'),
]:
    if path.exists():
        data = json.loads(path.read_text())
        summary[name] = {
            'fp32_map_50_95': data['fp32']['map_50_95'],
            'int8_map_50_95': data['int8']['map_50_95'],
            'map_delta': data['delta']['map_50_95'],
            'fp32_latency_ms': data['fp32']['avg_inference_ms_per_image'],
            'int8_latency_ms': data['int8']['avg_inference_ms_per_image'],
            'speedup': data['delta']['speedup'],
        }
SUMMARY_JSON = OUTPUT / 'pascal_voc_qat_summary.json'
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

labels = list(summary.keys())
if labels:
    int8_maps = [summary[label]['int8_map_50_95'] for label in labels]
    speedups = [summary[label]['speedup'] for label in labels]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(labels, int8_maps, color='tab:blue')
    axes[0].set_title('INT8 mAP@50:95 theo QAT epoch')
    axes[0].set_ylim(0, max(int8_maps + [0.01]) * 1.15)
    axes[0].grid(axis='y')
    axes[1].bar(labels, speedups, color='tab:orange')
    axes[1].set_title('INT8 speedup so với FP32 CPU')
    axes[1].grid(axis='y')
    plt.tight_layout()
    QAT_PLOT = OUTPUT / 'qat_epoch1_epoch2_benchmark.png'
    fig.savefig(QAT_PLOT, dpi=160)
    print('Saved QAT plot:', QAT_PLOT)
    plt.show()

print('Saved summary:', SUMMARY_JSON)
print_checkpoint_summary(OUTPUT)

In [ ]:
# Cell 15 - Upload checkpoint/benchmark thành Kaggle Dataset mới hoặc version mới
import kagglehub

assert KAGGLE_USERNAME != 'YOUR_KAGGLE_USERNAME', 'Hãy điền KAGGLE_USERNAME trước khi upload'
print('Uploading to:', CHECKPOINT_DATASET)
print_checkpoint_summary(OUTPUT)
kagglehub.dataset_upload(
    CHECKPOINT_DATASET,
    str(OUTPUT),
    version_notes='Pascal VOC ConvNeXt FP32 1 epoch + selective QAT 1/2 epoch benchmarks',
)
print('Uploaded:', CHECKPOINT_DATASET)